In [ ]:
from google.colab import userdata
!pip install datasets faiss-cpu transformers torch -q

from google.colab import drive
drive.mount('/content/drive')

print("Loading MedQA-USMLE from HuggingFace...")
from datasets import load_dataset

dataset = load_dataset(
    "GBaker/MedQA-USMLE-4-options",
    trust_remote_code=True
)

print(f"\nDataset loaded successfully")
print(f"Splits: {list(dataset.keys())}")
for split in dataset.keys():
    print(f"  {split}: {len(dataset[split])} questions")

print(f"\nSample question:")
sample = dataset['test'][0]
for k, v in sample.items():
    print(f"  {k}: {str(v)[:200]}")

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Loading PubMedQA as retrieval corpus...")

pubmedqa = load_dataset(
    "pubmed_qa",
    "pqa_unlabeled",
    split="train",
    trust_remote_code=False
)

print(f"PubMedQA abstracts loaded: {len(pubmedqa)}")
print(f"Columns: {pubmedqa.column_names}")

sample_doc = pubmedqa[0]
print(f"\nSample document:")
print(f"  PMID: {sample_doc.get('pubid', 'N/A')}")
print(f"  Question: {str(sample_doc.get('question', ''))[:100]}")

context = sample_doc.get('context', {})
print(f"  Context keys: {list(context.keys()) if isinstance(context, dict) else type(context)}")
print(f"  Context sample: {str(context)[:200]}")

In [ ]:
import numpy as np

print("Extracting abstract texts...")

docs = []
pmids = []

for item in pubmedqa:
    try:

        context_texts = item['context']['contexts']
        full_text = ' '.join(context_texts)
        if len(full_text.strip()) > 50:
            docs.append(full_text[:512])
            pmids.append(item['pubid'])
    except:
        continue

print(f"Documents extracted: {len(docs)}")
print(f"Sample doc (first 200 chars): {docs[0][:200]}")

import pickle
with open('/content/drive/MyDrive/docs.pkl', 'wb') as f:
    pickle.dump((docs, pmids), f)
print("Saved to Drive")

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import faiss
import pickle

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
print("Model loaded")

def embed_texts(texts, batch_size=256):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )

        with torch.no_grad():
            output = model(**encoded)

            attention_mask = encoded['attention_mask']
            token_embeddings = output.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).float()
            embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            embeddings = embeddings / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(embeddings.numpy())

        if (i // batch_size) % 20 == 0:
            print(f"  Embedded {min(i+batch_size, len(texts))}/{len(texts)} docs...")

    return np.vstack(all_embeddings)

print("\nEmbedding 61k documents...")
doc_embeddings = embed_texts(docs, batch_size=256)
print(f"Embeddings shape: {doc_embeddings.shape}")

print("\nBuilding FAISS index...")
d = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(doc_embeddings)
print(f"FAISS index built. Total vectors: {index.ntotal}")

faiss.write_index(index, '/content/drive/MyDrive/pubmedqa_index.faiss')
np.save('/content/drive/MyDrive/doc_embeddings.npy', doc_embeddings)
print("Saved index and embeddings to Drive")

In [ ]:
import numpy as np
import faiss
import pickle
from transformers import AutoTokenizer, AutoModel
import torch

questions = dataset['test'].select(range(200))

print(f"Running retrieval on {len(questions)} MedQA questions...")

def embed_texts(texts, batch_size=64):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=128, return_tensors='pt'
        )
        with torch.no_grad():
            output = model(**encoded)
            attention_mask = encoded['attention_mask']
            token_embeddings = output.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).float()
            embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            embeddings = embeddings / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.numpy())
    return np.vstack(all_embeddings)

question_texts = [q['question'] for q in questions]
gold_answers   = [q['answer'] for q in questions]
answer_options = [q['options'] for q in questions]

print("Embedding questions...")
q_embeddings = embed_texts(question_texts)
print(f"Question embeddings shape: {q_embeddings.shape}")

k = 10
print(f"\nRetrieving top-{k} documents per question...")
distances, indices = index.search(q_embeddings, k)
print(f"Retrieval done. Indices shape: {indices.shape}")

print(f"\nSample retrieval for question 0:")
print(f"  Q: {question_texts[0][:100]}")
print(f"  Retrieved doc indices: {indices[0]}")
print(f"  Top doc: {docs[indices[0][0]][:150]}")

with open('/content/drive/MyDrive/retrieval_results.pkl', 'wb') as f:
    pickle.dump({
        'indices': indices,
        'distances': distances,
        'gold_answers': gold_answers,
        'answer_options': answer_options,
        'question_texts': question_texts
    }, f)
print("\nSaved retrieval results to Drive")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.covariance import GraphicalLassoCV
from scipy import stats

n_questions = len(indices)
all_doc_ids = indices.flatten()
unique_docs, doc_counts = np.unique(all_doc_ids, return_counts=True)

print(f"Total unique documents retrieved: {len(unique_docs)}")
print(f"Most retrieved docs (top 20 frequency):")
top_idx = np.argsort(doc_counts)[::-1][:20]
for i in top_idx:
    print(f"  Doc {unique_docs[i]}: retrieved {doc_counts[i]} times | {docs[unique_docs[i]][:80]}")

In [ ]:
top_200_docs = unique_docs[np.argsort(doc_counts)[::-1][:200]]
doc_to_col = {doc_id: col for col, doc_id in enumerate(top_200_docs)}

print(f"Using top 200 most-retrieved documents")

n_q = n_questions
n_d = 200
X = np.zeros((n_q, n_d), dtype=float)

for q_idx in range(n_q):
    for doc_id in indices[q_idx]:
        if doc_id in doc_to_col:
            X[q_idx, doc_to_col[doc_id]] = 1.0

print(f"Co-retrieval matrix shape: {X.shape}")
print(f"Matrix density: {X.mean():.4f} (fraction of 1s)")
print(f"Mean docs per question in top-200: {X.sum(axis=1).mean():.2f}")

print("\nRunning Graphical Lasso...")
X_jittered = X + np.random.normal(0, 0.01, X.shape)

try:
    from sklearn.covariance import GraphicalLassoCV
    glasso = GraphicalLassoCV(cv=3, max_iter=200, n_jobs=-1)
    glasso.fit(X_jittered)
    precision_matrix = glasso.precision_
    print(f"Graphical Lasso converged")
    print(f"Precision matrix shape: {precision_matrix.shape}")

    prec_offdiag = precision_matrix.copy()
    np.fill_diagonal(prec_offdiag, 0)

    threshold = np.percentile(np.abs(prec_offdiag), 95)
    adjacency = (np.abs(prec_offdiag) > threshold).astype(float)
    node_degree = adjacency.sum(axis=1)

    print(f"\nEdge threshold (95th pctile): {threshold:.4f}")
    print(f"Mean node degree: {node_degree.mean():.2f}")
    print(f"Max node degree (hub): {node_degree.max():.0f}")
    print(f"Top 5 hub documents:")
    top_hubs = np.argsort(node_degree)[::-1][:5]
    for h in top_hubs:
        doc_id = top_200_docs[h]
        print(f"  Degree {node_degree[h]:.0f}: {docs[doc_id][:100]}")

except Exception as e:
    print(f"Graphical Lasso error: {e}")
    print("Falling back to correlation-based approach...")
    corr_matrix = np.corrcoef(X.T)
    np.fill_diagonal(corr_matrix, 0)
    node_degree = np.abs(corr_matrix).sum(axis=1)
    precision_matrix = corr_matrix
    adjacency = (np.abs(corr_matrix) > np.percentile(np.abs(corr_matrix), 95)).astype(float)
    node_degree = adjacency.sum(axis=1)
    print(f"Correlation fallback: mean degree {node_degree.mean():.2f}, max {node_degree.max():.0f}")

In [ ]:
from transformers import pipeline
import numpy as np
from scipy import stats

print("Loading QA model...")
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

qa_tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2")
qa_tokenizer.pad_token = qa_tokenizer.eos_token

print("Answering 200 MedQA questions...")

def get_model_answer(question, options):
    """Simple prompt to get A/B/C/D answer"""
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
    prompt = f"""Question: {question}

Options:
{opts_text}

The answer is (A/B/C/D):"""

    inputs = qa_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = qa_tokenizer.decode(
            inputs['input_ids'][0][-1:],
            skip_special_tokens=True
        )

    return None

print("Computing per-question hub scores...")
question_hub_scores = []

for q_idx in range(n_questions):
    retrieved_for_q = indices[q_idx]

    degrees_for_q = []
    for doc_id in retrieved_for_q:
        if doc_id in doc_to_col:
            col = doc_to_col[doc_id]
            degrees_for_q.append(node_degree[col])
        else:
            degrees_for_q.append(0.0)

    hub_score = max(degrees_for_q) if degrees_for_q else 0.0
    question_hub_scores.append(hub_score)

question_hub_scores = np.array(question_hub_scores)

print(f"Hub scores computed for {len(question_hub_scores)} questions")
print(f"Hub score distribution:")
print(f"  Mean: {question_hub_scores.mean():.3f}")
print(f"  Std:  {question_hub_scores.std():.3f}")
print(f"  Min:  {question_hub_scores.min():.3f}")
print(f"  Max:  {question_hub_scores.max():.3f}")
print(f"  Questions with hub score > 0: {(question_hub_scores > 0).sum()}")
print(f"  Questions with hub score = 0: {(question_hub_scores == 0).sum()}")
print(f"\nHub score value counts:")
unique_scores, score_counts = np.unique(question_hub_scores, return_counts=True)
for s, c in zip(unique_scores, score_counts):
    print(f"  Score {s:.0f}: {c} questions")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import numpy as np

print("Loading flan-t5-base directly...")
t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
t5_model.eval()
print("Model loaded")

def get_answer(question, options):
    opts_text = " ".join([f"({k}) {v}" for k, v in options.items()])
    prompt = f"Question: {question}\nOptions: {opts_text}\nAnswer:"

    try:
        inputs = t5_tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )
        with torch.no_grad():
            outputs = t5_model.generate(
                **inputs,
                max_new_tokens=10
            )
        result = t5_tokenizer.decode(outputs[0], skip_special_tokens=True).strip().upper()
        for char in result:
            if char in ['A', 'B', 'C', 'D']:
                return char
        return None
    except:
        return None

print("Getting model answers for 200 questions...")
model_answers = []
correct_flags = []

for i, q in enumerate(questions):
    pred = get_answer(q['question'], q['options'])
    gold = q['answer_idx'].strip().upper()
    model_answers.append(pred)
    correct_flags.append(1 if pred == gold else 0)

    if (i+1) % 50 == 0:
        print(f"  {i+1}/200 | Accuracy: {np.mean(correct_flags):.3f}")

correct_flags = np.array(correct_flags)
wrong_flags = 1 - correct_flags

print(f"\nFinal accuracy: {correct_flags.mean():.3f}")
print(f"Correct: {correct_flags.sum()} | Wrong: {wrong_flags.sum()}")

In [ ]:
from scipy import stats
import numpy as np

print("="*50)
print("CRITICAL CORRELATION TEST")
print("="*50)

print(f"\nSample sizes:")
print(f"  Total questions: {len(question_hub_scores)}")
print(f"  Wrong answers:   {wrong_flags.sum()}")
print(f"  Correct answers: {correct_flags.sum()}")

corr, pval = stats.pointbiserialr(question_hub_scores, wrong_flags)
print(f"\nTest 1 — Point Biserial Correlation:")
print(f"  Correlation: {corr:.4f}")
print(f"  P-value:     {pval:.4f}")
print(f"  Significant: {'YES' if pval < 0.05 else 'NO'}")

wrong_hub  = question_hub_scores[wrong_flags == 1]
correct_hub = question_hub_scores[correct_flags == 1]
print(f"\nTest 2 — Mean Hub Score:")
print(f"  Wrong answers:   {wrong_hub.mean():.3f} (n={len(wrong_hub)})")
print(f"  Correct answers: {correct_hub.mean():.3f} (n={len(correct_hub)})")
t_stat, t_pval = stats.ttest_ind(wrong_hub, correct_hub)
print(f"  T-test p-value:  {t_pval:.4f}")
print(f"  Significant: {'YES' if t_pval < 0.05 else 'NO'}")

has_hub = question_hub_scores > 0
no_hub  = question_hub_scores == 0
wrong_rate_hub    = wrong_flags[has_hub].mean()
wrong_rate_no_hub = wrong_flags[~has_hub].mean()
print(f"\nTest 3 — Wrong Rate by Hub Presence:")
print(f"  Questions WITH hub doc:    wrong rate = {wrong_rate_hub:.3f} (n={has_hub.sum()})")
print(f"  Questions WITHOUT hub doc: wrong rate = {wrong_rate_no_hub:.3f} (n={(~has_hub).sum()})")
chi2, chi_pval = stats.chi2_contingency([
    [wrong_flags[has_hub].sum(),  correct_flags[has_hub].sum()],
    [wrong_flags[~has_hub].sum(), correct_flags[~has_hub].sum()]
])[:2]
print(f"  Chi-square p-value: {chi_pval:.4f}")
print(f"  Significant: {'YES' if chi_pval < 0.05 else 'NO'}")

print(f"\n{'='*50}")
print("VERDICT")
print("="*50)
any_sig = (pval < 0.05) or (t_pval < 0.05) or (chi_pval < 0.05)
print(f"Any significant correlation found: {'YES — PAPER IS VIABLE' if any_sig else 'NO — SIGNAL DOES NOT EXIST'}")

In [ ]:
import sys

checks = {
    'dataset': 'dataset' in dir(),
    'index': 'index' in dir(),
    'docs': 'docs' in dir(),
    't5_model': 't5_model' in dir(),
    't5_tokenizer': 't5_tokenizer' in dir(),
}

print("Session state check:")
for name, exists in checks.items():
    status = "✓ LOADED" if exists else "✗ MISSING"
    print(f"  {name}: {status}")

import os
drive_files = [
    '/content/drive/MyDrive/docs.pkl',
    '/content/drive/MyDrive/pubmedqa_index.faiss',
    '/content/drive/MyDrive/doc_embeddings.npy',
]

print("\nDrive files:")
for f in drive_files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) / 1e6 if exists else 0
    status = f"✓ EXISTS ({size:.1f} MB)" if exists else "✗ MISSING"
    print(f"  {f.split('/')[-1]}: {status}")

In [ ]:
import torch
import numpy as np
import json
import os

def get_answer_with_prob(question, options):
    """
    Get model answer and probability for each option.
    Returns predicted letter and dict of option probabilities.
    """
    opts_text = " ".join([f"({k}) {v}" for k, v in options.items()])
    prompt = f"Question: {question}\nOptions: {opts_text}\nAnswer:"

    try:
        inputs = t5_tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = t5_model.generate(
                **inputs,
                max_new_tokens=10
            )

        result = t5_tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        ).strip().upper()

        pred = None
        for char in result:
            if char in ['A', 'B', 'C', 'D']:
                pred = char
                break

        return pred, result

    except Exception as e:
        return None, str(e)

print("Running flan-t5-base on all 1,273 MedQA test questions...")
print("This will take approximately 2 hours. Progress saved every 100 questions.\n")

save_path = '/content/drive/MyDrive/flan_t5_results.json'
if os.path.exists(save_path):
    with open(save_path, 'r') as f:
        results = json.load(f)
    print(f"Resuming from {len(results)} completed questions...")
else:
    results = []

all_questions = dataset['test']
start_idx = len(results)

for i in range(start_idx, len(all_questions)):
    q = all_questions[i]

    pred, raw_output = get_answer_with_prob(
        q['question'],
        q['options']
    )

    gold = q['answer_idx'].strip().upper()

    results.append({
        'idx': i,
        'question': q['question'][:100],
        'gold': gold,
        'pred': pred,
        'raw_output': raw_output,
        'correct': 1 if pred == gold else 0
    })

    if (i + 1) % 100 == 0:
        with open(save_path, 'w') as f:
            json.dump(results, f)

        acc = np.mean([r['correct'] for r in results])
        print(f"  {i+1}/1273 | Accuracy: {acc:.3f} | Saved to Drive")

with open(save_path, 'w') as f:
    json.dump(results, f)

correct = sum(r['correct'] for r in results)
total = len(results)
acc = correct / total

print(f"\nDone.")
print(f"Total: {total} questions")
print(f"Correct: {correct}")
print(f"Accuracy: {acc:.3f}")
print(f"Wrong: {total - correct}")
print(f"Saved to Drive: {save_path}")

In [ ]:
import requests
import json
import os
import time

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

def get_llama_answer(question, options, api_key):
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
    prompt = f"""You are a medical expert. Answer this USMLE question by selecting the single best answer.

Question: {question}

Options:
{opts_text}

Reply with only the letter (A, B, C, or D) of the correct answer."""

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 5,
        "temperature": 0.0
    }

    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=30
        )
        result = r.json()
        text = result['choices'][0]['message']['content'].strip().upper()
        for char in text:
            if char in ['A', 'B', 'C', 'D']:
                return char
        return None
    except Exception as e:
        return None

save_path = '/content/drive/MyDrive/llama_results.json'
if os.path.exists(save_path):
    with open(save_path, 'r') as f:
        results_llama = json.load(f)
    print(f"Resuming from {len(results_llama)} completed questions...")
else:
    results_llama = []

all_questions = dataset['test']
start_idx = len(results_llama)

print(f"Running Llama-3.1-8B on {len(all_questions) - start_idx} remaining questions...")

for i in range(start_idx, len(all_questions)):
    q = all_questions[i]

    pred = get_llama_answer(q['question'], q['options'], TOGETHER_API_KEY)
    gold = q['answer_idx'].strip().upper()

    results_llama.append({
        'idx': i,
        'gold': gold,
        'pred': pred,
        'correct': 1 if pred == gold else 0
    })

    if (i + 1) % 100 == 0:
        with open(save_path, 'w') as f:
            json.dump(results_llama, f)
        acc = sum(r['correct'] for r in results_llama) / len(results_llama)
        print(f"  {i+1}/1273 | Accuracy: {acc:.3f} | Saved")

    time.sleep(0.1)

with open(save_path, 'w') as f:
    json.dump(results_llama, f)

correct = sum(r['correct'] for r in results_llama)
total = len(results_llama)
print(f"\nLlama-3.1-8B Done.")
print(f"Accuracy: {correct/total:.3f}")
print(f"Correct: {correct} | Wrong: {total-correct}")

In [ ]:
import requests

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

r = requests.get(
    "https://api.together.xyz/v1/models",
    headers={"Authorization": f"Bearer {TOGETHER_API_KEY}"}
)

models = r.json()

print("Available Llama/Mistral models:")
for m in models:
    name = m.get('id', '')
    if any(x in name.lower() for x in ['llama', 'mistral', 'mixtral']):
        print(f"  {name}")

In [ ]:
import requests

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

q = dataset['test'][0]
opts_text = "\n".join([f"{k}: {v}" for k, v in q['options'].items()])
prompt = f"""You are a medical expert. Answer this USMLE question by selecting the single best answer.

Question: {q['question']}

Options:
{opts_text}

Reply with only the letter (A, B, C, or D) of the correct answer."""

r = requests.post(
    "https://api.together.xyz/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {TOGETHER_API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 5,
        "temperature": 0.0
    },
    timeout=30
)

print(f"Status: {r.status_code}")
data = r.json()
print(f"Full response: {json.dumps(data, indent=2)}")

if r.status_code == 200:
    raw_text = data['choices'][0]['message']['content']
    print(f"\nRaw model output: '{raw_text}'")
    print(f"Characters: {[c for c in raw_text]}")

In [ ]:
import requests

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

r = requests.get(
    "https://api.together.xyz/v1/models",
    headers={"Authorization": f"Bearer {TOGETHER_API_KEY}"}
)

if r.status_code == 200:
    print("KEY WORKS")
    print(f"Number of models available: {len(r.json())}")
elif r.status_code == 401:
    print("KEY INVALID - wrong key pasted")
else:
    print(f"Other error: {r.status_code}")
    print(r.json())

In [ ]:
import requests
import json

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

q = dataset['test'][0]
opts_text = "\n".join([f"{k}: {v}" for k, v in q['options'].items()])
prompt = f"""Answer this USMLE medical question. Reply with only A, B, C, or D.

Question: {q['question']}

A: {q['options']['A']}
B: {q['options']['B']}
C: {q['options']['C']}
D: {q['options']['D']}

Answer:"""

models_to_try = [
    "meta-llama/Meta-Llama-3-8B-Instruct-Lite",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "meta-llama/Llama-3-8b-chat-hf",
    "meta-llama/Llama-3.2-3B-Instruct",
]

working_model = None

for model in models_to_try:
    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {TOGETHER_API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 5,
                "temperature": 0.0
            },
            timeout=30
        )

        data = r.json()

        if r.status_code == 200:
            raw = data['choices'][0]['message']['content'].strip()
            print(f"✓ WORKS: {model}")
            print(f"  Raw output: '{raw}'")
            print(f"  Gold answer: {q['answer_idx']}")
            working_model = model
            break
        else:
            msg = data.get('error',{}).get('message','')[:80]
            print(f"✗ {model}: {msg}")

    except Exception as e:
        print(f"✗ {model}: {e}")

print(f"\nWorking model: {working_model}")

In [ ]:
import requests
import json
import os
import time

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')
WORKING_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct-Lite"

def get_answer(question, options, api_key):
    prompt = f"""Answer this USMLE medical question. Reply with only the single letter A, B, C, or D. Nothing else.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Single letter answer:"""

    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": WORKING_MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 3,
                "temperature": 0.0
            },
            timeout=30
        )
        result = r.json()
        text = result['choices'][0]['message']['content'].strip().upper()

        for char in text:
            if char in ['A', 'B', 'C', 'D']:
                return char
        return None
    except:
        return None

print("Testing on 5 questions...")
for i in range(5):
    q = dataset['test'][i]
    pred = get_answer(q['question'], q['options'], TOGETHER_API_KEY)
    gold = q['answer_idx'].strip().upper()
    correct = "✓" if pred == gold else "✗"
    print(f"  Q{i+1}: pred={pred} gold={gold} {correct}")

print("\nIf results look good, running all 1273...")

In [ ]:
import requests
import json
import os
import time

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')
WORKING_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct-Lite"

def get_answer(question, options, api_key):
    prompt = f"""Answer this USMLE medical question. Reply with only the single letter A, B, C, or D. Nothing else.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Single letter answer:"""

    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": WORKING_MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 3,
                "temperature": 0.0
            },
            timeout=30
        )
        result = r.json()
        text = result['choices'][0]['message']['content'].strip().upper()
        for char in text:
            if char in ['A', 'B', 'C', 'D']:
                return char
        return None
    except:
        return None

save_path = '/content/drive/MyDrive/llama_results.json'
if os.path.exists(save_path):
    os.remove(save_path)

results_llama = []
all_questions = dataset['test']

print(f"Running {WORKING_MODEL} on all 1273 questions...")
print("Saving to Drive every 100 questions.\n")

for i in range(len(all_questions)):
    q = all_questions[i]
    pred = get_answer(q['question'], q['options'], TOGETHER_API_KEY)
    gold = q['answer_idx'].strip().upper()

    results_llama.append({
        'idx': i,
        'gold': gold,
        'pred': pred,
        'correct': 1 if pred == gold else 0
    })

    if (i + 1) % 100 == 0:
        with open(save_path, 'w') as f:
            json.dump(results_llama, f)
        acc = sum(r['correct'] for r in results_llama) / len(results_llama)
        print(f"  {i+1}/1273 | Accuracy: {acc:.3f} | Saved")

    time.sleep(0.1)

with open(save_path, 'w') as f:
    json.dump(results_llama, f)

correct = sum(r['correct'] for r in results_llama)
total = len(results_llama)
print(f"\nDone.")
print(f"Model: {WORKING_MODEL}")
print(f"Accuracy: {correct/total:.3f}")
print(f"Correct: {correct} | Wrong: {total-correct}")

In [ ]:
import numpy as np
import json

with open('/content/drive/MyDrive/flan_t5_results.json', 'r') as f:
    flan_results = json.load(f)

with open('/content/drive/MyDrive/llama_results.json', 'r') as f:
    llama_results = json.load(f)

print(f"Flan-t5 results: {len(flan_results)} questions")
print(f"Llama results: {len(llama_results)} questions")

n_questions = 1273

flan_failures = np.array([1 - r['correct'] for r in flan_results], dtype=float)
llama_failures = np.array([1 - r['correct'] for r in llama_results], dtype=float)

print(f"\nFlan-t5 wrong: {flan_failures.sum():.0f} / {n_questions}")
print(f"Llama wrong:   {llama_failures.sum():.0f} / {n_questions}")

F = np.vstack([flan_failures, llama_failures])
print(f"\nFailure matrix F shape: {F.shape}")

E = F.T @ F / F.shape[0]
print(f"Error correlation matrix E shape: {E.shape}")

print("\nComputing SVD...")
U, sigma, Vt = np.linalg.svd(E)

print(f"\nTop 20 singular values:")
for i, s in enumerate(sigma[:20]):
    pct = s / sigma.sum() * 100
    print(f"  σ{i+1}: {s:.4f} ({pct:.1f}% of total variance)")

sigma_norm = sigma / sigma.sum()
entropy = -np.sum(sigma_norm * np.log(sigma_norm + 1e-10))
effective_rank = np.exp(entropy)
print(f"\nEffective rank: {effective_rank:.1f}")
print(f"Total possible rank: {min(F.shape)}")
print(f"Rank ratio: {effective_rank/min(F.shape):.3f}")

for k in [1, 2, 3, 5, 10, 20]:
    pct = sigma[:k].sum() / sigma.sum() * 100
    print(f"  Top {k:2d} components explain: {pct:.1f}% of variance")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np

files = [
    '/content/drive/MyDrive/flan_t5_results.json',
    '/content/drive/MyDrive/llama_results.json',
    '/content/drive/MyDrive/docs.pkl',
    '/content/drive/MyDrive/pubmedqa_index.faiss',
]

print("Checking Drive files:")
for f in files:
    exists = os.path.exists(f)
    size = os.path.getsize(f)/1e6 if exists else 0
    print(f"  {'✓' if exists else '✗'} {f.split('/')[-1]} ({size:.1f} MB)")

if os.path.exists('/content/drive/MyDrive/flan_t5_results.json'):
    with open('/content/drive/MyDrive/flan_t5_results.json') as f:
        flan = json.load(f)
    with open('/content/drive/MyDrive/llama_results.json') as f:
        llama = json.load(f)

    flan_fail  = np.array([1 - r['correct'] for r in flan])
    llama_fail = np.array([1 - r['correct'] for r in llama])

    both_fail       = ((flan_fail==1) & (llama_fail==1)).sum()
    both_correct    = ((flan_fail==0) & (llama_fail==0)).sum()
    flan_only_fail  = ((flan_fail==1) & (llama_fail==0)).sum()
    llama_only_fail = ((flan_fail==0) & (llama_fail==1)).sum()

    print(f"\nQuestion breakdown:")
    print(f"  Both correct:       {both_correct:4d} ({both_correct/1273*100:.1f}%)")
    print(f"  Both wrong:         {both_fail:4d} ({both_fail/1273*100:.1f}%)")
    print(f"  Only Flan wrong:    {flan_only_fail:4d} ({flan_only_fail/1273*100:.1f}%)")
    print(f"  Only Llama wrong:   {llama_only_fail:4d} ({llama_only_fail/1273*100:.1f}%)")
    print(f"\nCOGNITIVE TRAP candidates: {llama_only_fail}")

In [ ]:
from datasets import load_dataset
import json
import numpy as np

dataset = load_dataset("GBaker/MedQA-USMLE-4-options")

with open('/content/drive/MyDrive/flan_t5_results.json') as f:
    flan = json.load(f)
with open('/content/drive/MyDrive/llama_results.json') as f:
    llama = json.load(f)

flan_fail  = np.array([1 - r['correct'] for r in flan])
llama_fail = np.array([1 - r['correct'] for r in llama])

trap_indices = np.where((flan_fail==0) & (llama_fail==1))[0]

print(f"Total cognitive trap questions: {len(trap_indices)}")
print(f"\nFirst 10 cognitive trap questions:\n")

for i, idx in enumerate(trap_indices[:10]):
    q = dataset['test'][int(idx)]
    flan_pred = flan[idx]['pred']
    llama_pred = llama[idx]['pred']
    gold = q['answer_idx']

    print(f"{'='*60}")
    print(f"Question {i+1} (index {idx}):")
    print(f"  {q['question'][:200]}")
    print(f"  Options:")
    for k, v in q['options'].items():
        marker = " <- GOLD" if k == gold else ""
        marker += " <- LLAMA" if k == llama_pred else ""
        marker += " <- FLAN" if k == flan_pred else ""
        print(f"    {k}: {v[:80]}{marker}")
    print(f"  Gold: {gold} | Llama chose: {llama_pred} | Flan chose: {flan_pred}")
    print()

In [ ]:
import numpy as np
import json
from datasets import load_dataset

trap_indices = np.where((flan_fail==0) & (llama_fail==1))[0]

trap_analysis = []

for idx in trap_indices:
    q = dataset['test'][int(idx)]
    gold = q['answer_idx'].strip().upper()
    llama_pred = llama[int(idx)]['pred']
    flan_pred = flan[int(idx)]['pred']

    gold_text = q['options'].get(gold, '')
    wrong_text = q['options'].get(llama_pred, '') if llama_pred else ''
    question_text = q['question']

    trap_analysis.append({
        'idx': int(idx),
        'question': question_text,
        'gold': gold,
        'gold_text': gold_text,
        'llama_chose': llama_pred,
        'llama_wrong_text': wrong_text,
        'flan_chose': flan_pred,
        'topic': q.get('meta_info', 'unknown')
    })

print(f"Total trap questions: {len(trap_analysis)}")

topics = {}
for t in trap_analysis:
    topic = t['topic']
    topics[topic] = topics.get(topic, 0) + 1

print(f"\nTopic breakdown:")
for topic, count in sorted(topics.items(), key=lambda x: -x[1]):
    print(f"  {topic}: {count} ({count/len(trap_analysis)*100:.1f}%)")

print(f"\nOverall topic distribution in test set:")
all_topics = {}
for q in dataset['test']:
    t = q.get('meta_info', 'unknown')
    all_topics[t] = all_topics.get(t, 0) + 1
for topic, count in sorted(all_topics.items(), key=lambda x: -x[1]):
    trap_count = topics.get(topic, 0)
    trap_rate = trap_count / count * 100
    print(f"  {topic}: {count} total, {trap_count} traps ({trap_rate:.1f}% trap rate)")

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel

print("Loading embedding model...")
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Loaded")

def get_similarity(text1, text2):
    emb1 = embedder.encode([text1])
    emb2 = embedder.encode([text2])
    return cosine_similarity(emb1, emb2)[0][0]

print("\nAnalyzing 151 trap questions...")
print("Computing semantic similarities...\n")

results = []
for t in trap_analysis[:151]:
    q_text = t['question']
    gold_text = t['gold_text']
    wrong_text = t['llama_wrong_text']

    if not wrong_text or not gold_text:
        continue

    sim_wrong = get_similarity(q_text, wrong_text)
    sim_gold  = get_similarity(q_text, gold_text)

    results.append({
        'idx': t['idx'],
        'sim_wrong': float(sim_wrong),
        'sim_gold': float(sim_gold),
        'wrong_more_similar': sim_wrong > sim_gold,
        'similarity_gap': float(sim_wrong - sim_gold)
    })

wrong_more_similar = sum(r['wrong_more_similar'] for r in results)
total = len(results)
avg_gap = np.mean([r['similarity_gap'] for r in results])

print(f"Results for {total} trap questions:")
print(f"\nLlama's wrong answer is MORE similar to question than gold answer:")
print(f"  {wrong_more_similar}/{total} = {wrong_more_similar/total*100:.1f}%")
print(f"\nAverage similarity gap (wrong - gold):")
print(f"  {avg_gap:.4f} (positive = wrong answer more similar to question)")

non_trap_indices = np.where((flan_fail==1) & (llama_fail==0))[0]
print(f"\nComputing same metric for {len(non_trap_indices)} non-trap questions...")

non_trap_results = []
for idx in non_trap_indices[:151]:
    q = dataset['test'][int(idx)]
    gold = q['answer_idx'].strip().upper()
    llama_pred = llama[int(idx)]['pred']

    gold_text = q['options'].get(gold, '')
    wrong_text = q['options'].get(llama_pred, '') if llama_pred else ''
    q_text = q['question']

    if not wrong_text or not gold_text:
        continue

    sim_wrong = get_similarity(q_text, wrong_text)
    sim_gold  = get_similarity(q_text, gold_text)

    non_trap_results.append({
        'sim_wrong': float(sim_wrong),
        'sim_gold': float(sim_gold),
        'wrong_more_similar': sim_wrong > sim_gold,
        'similarity_gap': float(sim_wrong - sim_gold)
    })

non_trap_wrong_similar = sum(r['wrong_more_similar'] for r in non_trap_results)
non_trap_avg_gap = np.mean([r['similarity_gap'] for r in non_trap_results])

print(f"\nFor NON-trap questions (Flan wrong, Llama correct):")
print(f"  Wrong answer more similar: {non_trap_wrong_similar}/{len(non_trap_results)} = {non_trap_wrong_similar/len(non_trap_results)*100:.1f}%")
print(f"  Average similarity gap: {non_trap_avg_gap:.4f}")

print(f"\n{'='*50}")
print(f"COMPARISON:")
print(f"  Trap questions — wrong answer more similar:     {wrong_more_similar/total*100:.1f}%")
print(f"  Non-trap questions — wrong answer more similar: {non_trap_wrong_similar/len(non_trap_results)*100:.1f}%")
print(f"\n  If trap % >> non-trap %: SEDUCTIVE DISTRACTOR hypothesis confirmed")
print(f"  If similar: effect is not about semantic similarity")

In [ ]:
import numpy as np
from scipy import stats

both_wrong_indices = np.where((flan_fail==1) & (llama_fail==1))[0]

print(f"Computing similarities for 'both wrong' baseline questions...")
print(f"Both wrong: {len(both_wrong_indices)} questions")

both_wrong_results = []
for idx in both_wrong_indices[:151]:
    q = dataset['test'][int(idx)]
    gold = q['answer_idx'].strip().upper()
    llama_pred = llama[int(idx)]['pred']

    gold_text = q['options'].get(gold, '')
    wrong_text = q['options'].get(llama_pred, '') if llama_pred else ''
    q_text = q['question']

    if not wrong_text or not gold_text or wrong_text == gold_text:
        continue

    sim_wrong = get_similarity(q_text, wrong_text)
    sim_gold  = get_similarity(q_text, gold_text)

    both_wrong_results.append({
        'sim_wrong': float(sim_wrong),
        'sim_gold': float(sim_gold),
        'similarity_gap': float(sim_wrong - sim_gold)
    })

print("\nTesting plausibility hypothesis...")
print("(Are Llama's wrong answers more similar to the CORRECT answer?)")

trap_wrong_to_gold = []
both_wrong_to_gold = []

for t in trap_analysis[:149]:
    gold_text = t['gold_text']
    wrong_text = t['llama_wrong_text']
    if gold_text and wrong_text:
        sim = get_similarity(wrong_text, gold_text)
        trap_wrong_to_gold.append(float(sim))

for idx in both_wrong_indices[:149]:
    q = dataset['test'][int(idx)]
    gold = q['answer_idx'].strip().upper()
    llama_pred = llama[int(idx)]['pred']
    gold_text = q['options'].get(gold, '')
    wrong_text = q['options'].get(llama_pred, '') if llama_pred else ''
    if gold_text and wrong_text and wrong_text != gold_text:
        sim = get_similarity(wrong_text, gold_text)
        both_wrong_to_gold.append(float(sim))

trap_mean = np.mean(trap_wrong_to_gold)
both_mean = np.mean(both_wrong_to_gold)
t_stat, p_val = stats.ttest_ind(trap_wrong_to_gold, both_wrong_to_gold)

print(f"\nSimilarity between wrong answer and correct answer:")
print(f"  Cognitive trap questions: {trap_mean:.4f}")
print(f"  Both-wrong questions:     {both_mean:.4f}")
print(f"  T-test p-value: {p_val:.4f}")
print(f"  Significant: {'YES' if p_val < 0.05 else 'NO'}")
print(f"\n  If trap > both-wrong: Llama's wrong answers are MORE plausible")
print(f"  This would confirm the seductive distractor hypothesis")

trap_lengths = [len(trap_analysis[i]['question'].split())
                for i in range(len(trap_analysis))]
both_wrong_lengths = [len(dataset['test'][int(idx)]['question'].split())
                      for idx in both_wrong_indices[:151]]
both_correct_indices = np.where((flan_fail==0) & (llama_fail==0))[0]
both_correct_lengths = [len(dataset['test'][int(idx)]['question'].split())
                        for idx in both_correct_indices[:151]]

print(f"\nQuestion length analysis:")
print(f"  Cognitive trap questions: {np.mean(trap_lengths):.1f} words avg")
print(f"  Both-wrong questions:     {np.mean(both_wrong_lengths):.1f} words avg")
print(f"  Both-correct questions:   {np.mean(both_correct_lengths):.1f} words avg")

t2, p2 = stats.ttest_ind(trap_lengths, both_wrong_lengths)
print(f"  Trap vs both-wrong p-value: {p2:.4f}")
print(f"  Significant: {'YES' if p2 < 0.05 else 'NO'}")

In [ ]:
import requests
import json
import time

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

def ask_70b(question, options, gold, wrong_answer, api_key):
    gold_text = options.get(gold, '')
    wrong_text = options.get(wrong_answer, '')
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are an expert medical educator analyzing why a medical AI model chose the wrong answer on a USMLE question.

Question: {question}

Options:
{opts_text}

Correct answer: {gold} - {gold_text}
Wrong answer chosen by AI: {wrong_answer} - {wrong_text}

In 2-3 sentences explain: What cognitive bias or reasoning error made the wrong answer seductive?
Be specific about the mechanism (anchoring, availability bias, framing effect, etc.)."""

    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 150,
                "temperature": 0.0
            },
            timeout=30
        )
        return r.json()['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f"Error: {e}"

print("Asking Llama-3.3-70B to explain 10 cognitive traps...\n")

explanations = []

for i, t in enumerate(trap_analysis[:10]):
    q = dataset['test'][t['idx']]

    explanation = ask_70b(
        t['question'],
        q['options'],
        t['gold'],
        t['llama_chose'],
        TOGETHER_API_KEY
    )

    explanations.append({
        'idx': t['idx'],
        'explanation': explanation
    })

    print(f"Question {i+1} (index {t['idx']}):")
    print(f"  Q: {t['question'][:100]}...")
    print(f"  Gold: {t['gold']} | Llama chose: {t['llama_chose']}")
    print(f"  Explanation: {explanation}")
    print()

    time.sleep(0.5)

with open('/content/drive/MyDrive/trap_explanations.json', 'w') as f:
    json.dump(explanations, f)
print("Saved to Drive")

In [ ]:
import requests
import json
import time

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

def classify_bias(question, options, gold, wrong_answer, api_key):
    gold_text = options.get(gold, '')
    wrong_text = options.get(wrong_answer, '')
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are an expert medical educator. Classify the cognitive bias that caused this AI model error.

Question: {question}

Options:
{opts_text}

Correct answer: {gold} - {gold_text}
Wrong answer chosen: {wrong_answer} - {wrong_text}

Reply with ONLY one of these labels:
- AVAILABILITY_BIAS (over-weighting salient/memorable symptoms)
- ANCHORING_BIAS (over-relying on first piece of information)
- FRAMING_EFFECT (different conclusion from same info presented differently)
- PREMATURE_CLOSURE (stopping reasoning too early)
- OTHER

Single label only, no explanation:"""

    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 10,
                "temperature": 0.0
            },
            timeout=30
        )
        text = r.json()['choices'][0]['message']['content'].strip().upper()

        for label in ['AVAILABILITY_BIAS', 'ANCHORING_BIAS',
                      'FRAMING_EFFECT', 'PREMATURE_CLOSURE', 'OTHER']:
            if label in text:
                return label
        return 'OTHER'
    except:
        return 'ERROR'

print("Classifying bias type for all 151 trap questions...")
print("Cost estimate: ~$0.30\n")

bias_labels = []

for i, t in enumerate(trap_analysis):
    q = dataset['test'][t['idx']]

    label = classify_bias(
        t['question'],
        q['options'],
        t['gold'],
        t['llama_chose'],
        TOGETHER_API_KEY
    )

    bias_labels.append({
        'idx': t['idx'],
        'bias_type': label
    })

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/151 done...")

    time.sleep(0.3)

with open('/content/drive/MyDrive/bias_labels.json', 'w') as f:
    json.dump(bias_labels, f)

from collections import Counter
counts = Counter(b['bias_type'] for b in bias_labels)
total = len(bias_labels)

print(f"\nBias classification results ({total} questions):")
for bias, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {bias}: {count} ({count/total*100:.1f}%)")

print("\nSaved to Drive")

In [ ]:
import requests
import json
import os
import time

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

def get_answer(question, options, api_key, model):
    prompt = f"""Answer this USMLE medical question. Reply with only the single letter A, B, C, or D.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Single letter answer:"""

    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 3,
                "temperature": 0.0
            },
            timeout=30
        )
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for char in text:
            if char in ['A', 'B', 'C', 'D']:
                return char
        return None
    except:
        return None

save_path = '/content/drive/MyDrive/mistral_results.json'
if os.path.exists(save_path):
    os.remove(save_path)

results_mistral = []
all_questions = dataset['test']
MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Running {MODEL} on 1273 questions...")

for i in range(len(all_questions)):
    q = all_questions[i]
    pred = get_answer(q['question'], q['options'], TOGETHER_API_KEY, MODEL)
    gold = q['answer_idx'].strip().upper()

    results_mistral.append({
        'idx': i,
        'gold': gold,
        'pred': pred,
        'correct': 1 if pred == gold else 0
    })

    if (i + 1) % 100 == 0:
        with open(save_path, 'w') as f:
            json.dump(results_mistral, f)
        acc = sum(r['correct'] for r in results_mistral) / len(results_mistral)
        print(f"  {i+1}/1273 | Accuracy: {acc:.3f} | Saved")

    time.sleep(0.1)

with open(save_path, 'w') as f:
    json.dump(results_mistral, f)

correct = sum(r['correct'] for r in results_mistral)
total = len(results_mistral)
print(f"\nMistral Done.")
print(f"Accuracy: {correct/total:.3f}")
print(f"Correct: {correct} | Wrong: {total-correct}")

In [ ]:
import requests

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

q = dataset['test'][0]

prompt = f"""Answer this USMLE medical question. Reply with only the single letter A, B, C, or D.

Question: {q['question']}

A: {q['options']['A']}
B: {q['options']['B']}
C: {q['options']['C']}
D: {q['options']['D']}

Single letter answer:"""

r = requests.post(
    "https://api.together.xyz/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {TOGETHER_API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "mistralai/Mistral-7B-Instruct-v0.3",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 10,
        "temperature": 0.0
    },
    timeout=30
)

print(f"Status: {r.status_code}")
data = r.json()
print(f"Full response: {data}")
if r.status_code == 200:
    raw = data['choices'][0]['message']['content']
    print(f"Raw text: '{raw}'")
    print(f"Characters: {[c for c in raw]}")

In [ ]:
import requests

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

r = requests.get(
    "https://api.together.xyz/v1/models",
    headers={"Authorization": f"Bearer {TOGETHER_API_KEY}"}
)

models = r.json()

print("Serverless Mistral/different architecture models:")
for m in models:
    name = m.get('id', '')
    model_type = m.get('type', '')
    pricing = m.get('pricing', {})

    if any(x in name.lower() for x in ['mistral', 'mixtral', 'deepseek', 'qwen', 'gemma']):
        print(f"  {name} | type: {model_type}")

In [ ]:
import requests
import json
import os
import time

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

def get_answer(question, options, api_key, model):
    prompt = f"""Answer this USMLE medical question. Reply with only the single letter A, B, C, or D. Nothing else.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Single letter answer:"""

    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 3,
                "temperature": 0.0
            },
            timeout=30
        )
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for char in text:
            if char in ['A', 'B', 'C', 'D']:
                return char
        return None
    except:
        return None

print("Testing both models on 3 questions...\n")

test_models = [
    "mistralai/Mistral-7B-Instruct-v0.1",
    "Qwen/Qwen2.5-7B-Instruct-Turbo"
]

working_models = []

for model in test_models:
    print(f"Testing {model}...")
    correct = 0
    for i in range(3):
        q = dataset['test'][i]
        pred = get_answer(q['question'], q['options'], TOGETHER_API_KEY, model)
        gold = q['answer_idx'].strip().upper()
        status = "✓" if pred == gold else "✗"
        print(f"  Q{i+1}: pred={pred} gold={gold} {status}")

    if pred is not None:
        working_models.append(model)
        print(f"  STATUS: WORKING\n")
    else:
        print(f"  STATUS: FAILED\n")

print(f"Working models: {working_models}")

In [ ]:
import requests
import json
import os
import time

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')
MODEL = "Qwen/Qwen2.5-7B-Instruct-Turbo"

def get_answer(question, options, api_key, model):
    prompt = f"""Answer this USMLE medical question. Reply with only the single letter A, B, C, or D. Nothing else.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Single letter answer:"""

    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 3,
                "temperature": 0.0
            },
            timeout=30
        )
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for char in text:
            if char in ['A', 'B', 'C', 'D']:
                return char
        return None
    except:
        return None

save_path = '/content/drive/MyDrive/qwen_results.json'
if os.path.exists(save_path):
    os.remove(save_path)

results_qwen = []
all_questions = dataset['test']

print(f"Running Qwen2.5-7B on 1273 questions...")

for i in range(len(all_questions)):
    q = all_questions[i]
    pred = get_answer(q['question'], q['options'], TOGETHER_API_KEY, MODEL)
    gold = q['answer_idx'].strip().upper()

    results_qwen.append({
        'idx': i,
        'gold': gold,
        'pred': pred,
        'correct': 1 if pred == gold else 0
    })

    if (i + 1) % 100 == 0:
        with open(save_path, 'w') as f:
            json.dump(results_qwen, f)
        acc = sum(r['correct'] for r in results_qwen) / len(results_qwen)
        print(f"  {i+1}/1273 | Accuracy: {acc:.3f} | Saved")

    time.sleep(0.1)

with open(save_path, 'w') as f:
    json.dump(results_qwen, f)

correct = sum(r['correct'] for r in results_qwen)
total = len(results_qwen)
print(f"\nQwen Done.")
print(f"Accuracy: {correct/total:.3f}")
print(f"Correct: {correct} | Wrong: {total-correct}")

In [ ]:
import numpy as np
import json

with open('/content/drive/MyDrive/flan_t5_results.json') as f:
    flan = json.load(f)
with open('/content/drive/MyDrive/llama_results.json') as f:
    llama = json.load(f)
with open('/content/drive/MyDrive/qwen_results.json') as f:
    qwen = json.load(f)

flan_fail = np.array([1 - r['correct'] for r in flan])
llama_fail = np.array([1 - r['correct'] for r in llama])
qwen_fail  = np.array([1 - r['correct'] for r in qwen])

print("Model summary:")
print(f"  Flan-t5-base:   {flan_fail.mean()*100:.1f}% wrong ({flan_fail.sum():.0f}/1273)")
print(f"  Llama-3-8B:     {llama_fail.mean()*100:.1f}% wrong ({llama_fail.sum():.0f}/1273)")
print(f"  Qwen-2.5-7B:    {qwen_fail.mean()*100:.1f}% wrong ({qwen_fail.sum():.0f}/1273)")

llama_trap_idx = set(np.where((flan_fail==0) & (llama_fail==1))[0])
print(f"\nOriginal Llama trap questions: {len(llama_trap_idx)}")

qwen_also_fails = sum(1 for idx in llama_trap_idx if qwen_fail[idx] == 1)
qwen_also_correct = sum(1 for idx in llama_trap_idx if qwen_fail[idx] == 0)

print(f"\nOf the 151 Llama trap questions:")
print(f"  Qwen also fails:   {qwen_also_fails} ({qwen_also_fails/len(llama_trap_idx)*100:.1f}%)")
print(f"  Qwen gets correct: {qwen_also_correct} ({qwen_also_correct/len(llama_trap_idx)*100:.1f}%)")

qwen_trap_idx = set(np.where((flan_fail==0) & (qwen_fail==1))[0])
print(f"\nQwen trap questions (Qwen fails, Flan succeeds): {len(qwen_trap_idx)}")

overlap = llama_trap_idx & qwen_trap_idx
print(f"Overlap (both models trapped on same questions): {len(overlap)}")
print(f"Overlap rate: {len(overlap)/len(llama_trap_idx)*100:.1f}% of Llama traps")

print(f"\n{'='*50}")
print(f"KEY FINDING:")
print(f"  Questions that trap BOTH Llama AND Qwen")
print(f"  but Flan-t5 gets right: {len(overlap)}")
print(f"  These are architecture-agnostic cognitive traps")
print(f"{'='*50}")

categories = {
    'All correct': ((flan_fail==0) & (llama_fail==0) & (qwen_fail==0)).sum(),
    'All wrong': ((flan_fail==1) & (llama_fail==1) & (qwen_fail==1)).sum(),
    'Only Flan wrong': ((flan_fail==1) & (llama_fail==0) & (qwen_fail==0)).sum(),
    'Llama+Qwen wrong, Flan right': ((flan_fail==0) & (llama_fail==1) & (qwen_fail==1)).sum(),
    'Only Llama wrong': ((flan_fail==0) & (llama_fail==1) & (qwen_fail==0)).sum(),
    'Only Qwen wrong': ((flan_fail==0) & (llama_fail==0) & (qwen_fail==1)).sum(),
}

print(f"\nFull 3-model breakdown:")
for cat, count in categories.items():
    print(f"  {cat}: {count} ({count/1273*100:.1f}%)")